# UTS Graf Sains Data
**Nama :** Nadira Zahra Ramadhani

**NIM :** 23031554099

**Kelas :** 2023D


**Periode P1 (training) :** 17 Juli - 30 Juli  **Periode P2 (testing) :** 31 Juli - 6 Agustus

## 1. Tugas Pendahuluan
### 1.1 Persiapan Lingkungan

In [1]:
import sys
import neo4j
print("Python :", sys.version.split()[0])
print("Kernel :", sys.executable)
print("neo4j  :", neo4j.__version__)

Python : 3.11.9
Kernel : d:\Semester 7\Graph Sains Data\UTS GSD\myvenv\Scripts\python.exe
neo4j  : 6.4.1


In [2]:
from getpass import getpass
from neo4j import GraphDatabase, RoutingControl

URI = "neo4j://localhost:7687"
USER = "neo4j"
PASSWORD = getpass("Password Neo4j: ")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print("Terhubung ke Neo4j")

Terhubung ke Neo4j


### 1.2 Tutorial Driver Neo4j

In [3]:
def add_friend(driver, name, friend_name):
    driver.execute_query(
        "MERGE (a:Person {name: $name}) "
        "MERGE (friend:Person {name: $friend_name}) "
        "MERGE (a)-[:KNOWS]->(friend)",
        name=name, friend_name=friend_name, database_="neo4j",
    )

def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
        "RETURN friend.name ORDER BY friend.name",
        name=name, database_="neo4j", routing_=RoutingControl.READ,
    )
    for record in records:
        print(record["friend.name"])

for f in ["Guinevere", "Lancelot", "Merlin"]:
    add_friend(driver, "Arthur", f)
print_friends(driver, "Arthur")

Guinevere
Lancelot
Merlin


**Pertanyaan Pendahuluan**
1. Database yang digunakan adalah `neo4j` karena itu database default dari parameter `database_="neo4j"`.

2. Database juga bisa diganti dengan `USE namadb`, `:use namadb`, atau lewat `driver.session(database=...)`. Untuk melihat database bisa menggunakan `SHOW DATABASES`, sedangkan untuk membuat database baru menggunakan `CREATE DATABASE`.

3. Jenis grafnya directed, unweighted, dan homogeneous karena hanya ada satu label `Person` dan satu relasi `KNOWS`. Grafnya simple karena menggunakan `MERGE`, dengan Arthur sebagai pusat yang terhubung ke node lainnya.

In [4]:
driver.execute_query("MATCH (n) DETACH DELETE n", database_="neo4j")
print("Database dibersihkan")

Database dibersihkan


### 1.3 Impor Data RetailRocket

In [5]:
import os
import pandas as pd

csv_path = os.path.join("data", "events.csv")
assert os.path.exists(csv_path), f"{csv_path} tidak ditemukan"

pd.read_csv(csv_path, nrows=5, dtype={"transactionid": str})

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [7]:
q_nodes = "MATCH (n) RETURN labels(n) AS label, count(*) AS jumlah"
q_rels  = "MATCH ()-[r]->() RETURN type(r) AS relasi, count(*) AS jumlah"
for q in (q_nodes, q_rels):
    records, _, _ = driver.execute_query(q, database_="neo4j")
    for r in records:
        print(dict(r))

{'label': ['Visitor'], 'jumlah': 54913}
{'label': ['Item'], 'jumlah': 39559}
{'relasi': 'VIEWED', 'jumlah': 96754}
{'relasi': 'ADDED_TO_CART', 'jumlah': 2394}
{'relasi': 'BOUGHT', 'jumlah': 852}
